# 1. Import the necessary libraries

In [ ]:
# Adds the parent directory to sys.path
import sys
sys.path.append('../')

from problems.benchmark_problems import get_problem
from problems.microgrid_function import microgrid_function
from runners import run_cmopso, run_maco, run_mesh, run_mopso_cd, run_nsga2, run_nspso, run_spea2
from tuners import fine_tune_cmopso, fine_tune_maco, fine_tune_mesh, fine_tune_mopso_cd, fine_tune_nsga2, fine_tune_nspso, fine_tune_spea2

from contextlib import redirect_stdout
from io import StringIO
from joblib import Parallel, delayed
from optuna.pruners import NopPruner
from pymoo.indicators.hv import HV

import numpy as np

# 2. Define the general configuration of the experiments

In [ ]:
#################### CUSTOMIZABLE ####################

# Number of processes to execute the experiments in parallel
workers = 16

# Objective function configuration (function name, number of objectives, number of decision variables)
experiments = [
    ('zdt1', 2, 20, None), ('zdt2', 2, 20, None), ('zdt3', 2, 20, None), ('zdt4', 2, 20, None), ('zdt6', 2, 20, None),
    ('dtlz1', 3, 20, None), ('dtlz2', 3, 20, None), ('dtlz3', 3, 20, None), ('dtlz4', 3, 20, None), ('dtlz5', 3, 20, None), ('dtlz6', 3, 20, None), ('dtlz7', 3, 20, None),
    ('wfg1', 3, 20, 10), ('wfg2', 3, 20, 10), ('wfg3', 3, 20, 10), ('wfg4', 3, 20, 10), ('wfg5', 3, 20, 10), ('wfg6', 3, 20, 10), ('wfg7', 3, 20, 10), ('wfg8', 3, 20, 10),
    ('wfg9', 3, 20, 10)
]

# Execution configuration
num_runs = 30 # Number of runs
max_fitness_eval = 25600 # Maximum fitness evaluations (not used if it is None)
population_size = 128 # Population size
random_state = None # Set a seed for random numbers (not used if it is None)

results_folder = '../results' # Folder to store the results
tuning_folder = '../hyperparams' # Folder to get the tuned parameters

# Fine tuning configuration
n_trials = 500 # Number of trials for Optuna
n_steps = 10 # Number of rounds in a trial
optuna_pruner = NopPruner() # Optuna's pruner to prune bad trials according to a metric
# optuna_pruner = optuna.pruners.WilcoxonPruner(p_threshold=0.05, n_startup_steps=n_steps//2)
hypercube_vertex = 100 # Consider the vertex of the hypercube
######################################################

ref_point = [np.array([hypercube_vertex] * exp[1]) for exp in experiments] # Reference point for hypervolume calculation
indicators = [HV(ref_point=rf) for rf in ref_point] # Define the indicator to calculate the volume

# 3 Define auxiliar functions

In [ ]:
# Function to get the fitness function configuration
def get_exp_problem(func_name, objective_dim, position_dim, wfg_k):
	select_bat = {'Lead_Acid': 0, 'Li-ion': 1, 'ZEBRA': 2, 'NaS': 3, 'NiCd': 4, 'NiMH': 5, 'RFV': 6, 'ZnBr': 7}
	# Microgrid problem
	if func_name in select_bat:
		load = np.genfromtxt('../seasonal_data/load.txt')
		temperature = np.genfromtxt('../seasonal_data/temperature.txt')
		solar_data = np.genfromtxt('../seasonal_data/irradiance.txt')
		wind_data = np.genfromtxt('../seasonal_data/wind.txt')
		def microgrid_func(args):
			return microgrid_function(args[0], args[1], args[2], select_bat[func_name], load, temperature, solar_data, wind_data)
		lower_bound_array = np.array([1.0, 1.0, 1.0])
		upper_bound_array = np.array([2000.0, 2000.0, 2000.0])
		return {'fitness': microgrid_func,
		        'objective_dim': objective_dim,
		        'position_dim': position_dim,
		        'lower_bound_array': lower_bound_array,
		        'upper_bound_array': upper_bound_array}
	# Benchmark problems
	else:
		func, lower_bound_array, upper_bound_array = get_problem(func_name, n_var=position_dim, n_obj=objective_dim, wfg_k=wfg_k)
		return {'fitness': func,
		        'objective_dim': objective_dim,
		        'position_dim': position_dim,
		        'lower_bound_array': lower_bound_array,
		        'upper_bound_array': upper_bound_array}

# Function to capture errors during parallel execution without stopping the other processes
def safe_run(run_function, params):
	try:
		f = StringIO()
		with redirect_stdout(f):
			status = run_function(*params)
		return status, f.getvalue()
	except Exception as e:
		return f'Error: {str(e)}'

# 4. Run experiments with the algorithms (each problem in parallel)

## 4.1 MESH

In [ ]:
#################### CUSTOMIZABLE ####################
# Fixed parameters
mesh_memory_size = population_size # Maximum number of particles in memory


# Tunable parameters
mesh_global_best_type = 0 # 0 -> Sigma method (G1) | 1 -> Sigma method in fronts (G2)
mesh_differential_mutation_pool_type = 0 # 0 -> Sampling from memory (S1) | 1 -> Sampling from population (S2) | 2 -> Sampling from memory and population (S3)
mesh_differential_mutation_type = 0 # 0 -> DE\rand\1\Bin (D1) | 1 -> DE\rand\2\Bin (D2) | 2 -> DE/Best/1/Bin (D3) | 3 -> DE/Current-to-best/1/Bin (D4) | 4 -> DE/Current-to-rand/1/Bin (D5)
mesh_communication_probability = 0.8 # Communication probability
mesh_mutation_rate = 0.9 # Mutation rate
mesh_personal_guide_array_size = 1 # Number of personal guides
######################################################

### Fine Tuning

In [ ]:
# Set the list of parameters
mesh_fine_tuning_params_list = [[
    {
        'name': f'MESH_{exp[0]}_{exp[1]}_{exp[2]}',
        'fine_tuning_folder': tuning_folder,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
    },
    get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
    {
        'n_trials': n_trials,
        'n_steps': n_steps,
        'pruner': optuna_pruner
    },
    {
        'memory_size': mesh_memory_size,
    },
    indicators[i]
    ]
     for i, exp in enumerate(experiments)
]

# Execute in parallel
if __name__ == "__main__":
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(fine_tune_mesh, params)
        for params in mesh_fine_tuning_params_list
    )
    print(return_values)

### Running

In [ ]:
# Set the list of parameters
mesh_run_params_list = [[
    {
        'name': f'MESH_{exp[0]}_{exp[1]}_{exp[2]}',
        'results_folder': results_folder,
        'fine_tuning_folder': tuning_folder,
        'num_runs': num_runs,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
     },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
     {
        'memory_size': mesh_memory_size,
        'global_best_attribution_type': mesh_global_best_type,
        'differential_mutation_pool_type': mesh_differential_mutation_pool_type,
        'differential_mutation_type': mesh_differential_mutation_type,
        'communication_probability': mesh_communication_probability,
        'mutation_rate': mesh_mutation_rate,
        'personal_guide_array_size': mesh_personal_guide_array_size
      }
    ]
     for exp in experiments
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(run_mesh, params)
        for params in mesh_run_params_list
    )
    print(return_values)

## 4.2 CMOPSO

In [ ]:
#################### CUSTOMIZABLE ####################
# Tunable parameters
cmopso_max_velocity_rate = 0.5
cmopso_elite_size = population_size // 2
cmopso_initial_velocity = 'random'
cmopso_mutate_rate = 0.5
######################################################

### Fine Tuning

In [ ]:
# Set the list of parameters
cmopso_fine_tuning_params_list = [[
    {
        'name': f'CMOPSO_{exp[0]}_{exp[1]}_{exp[2]}',
        'fine_tuning_folder': tuning_folder,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
    },
    get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
    {
        'n_trials': n_trials,
        'n_steps': n_steps,
        'pruner': optuna_pruner
    },
    {},
    indicators[i]
    ]
     for i, exp in enumerate(experiments)
]

# Execute in parallel
if __name__ == "__main__":
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(fine_tune_cmopso, params)
        for params in cmopso_fine_tuning_params_list
    )
    print(return_values)

### Running

In [ ]:
# Set the list of parameters
cmopso_run_params_list = [[
    {
        'name': f'CMOPSO_{exp[0]}_{exp[1]}_{exp[2]}',
        'results_folder': results_folder,
        'fine_tuning_folder': tuning_folder,
        'num_runs': num_runs,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
     },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
     {
        'max_velocity_rate': cmopso_max_velocity_rate,
        'elite_size': cmopso_elite_size,
        'initial_velocity': cmopso_initial_velocity,
        'mutate_rate': cmopso_mutate_rate
     }
    ]
     for exp in experiments
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(run_cmopso, params)
        for params in cmopso_run_params_list
    )
    print(return_values)

## 4.3 MACO

In [ ]:
#################### CUSTOMIZABLE ####################
# Tunable parameters
maco_ker = population_size
maco_q = 0.5
maco_threshold = max_fitness_eval // (2 * population_size)
maco_n_gen_mark = max_fitness_eval // (2 * population_size)
maco_focus = 2.0
######################################################

### Fine Tuning

In [ ]:
# Set the list of parameters
maco_fine_tuning_params_list = [[
    {
        'name': f'MACO_{exp[0]}_{exp[1]}_{exp[2]}',
        'fine_tuning_folder': tuning_folder,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
    },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
	 {
        'n_trials': n_trials,
        'n_steps': n_steps,
        'pruner': optuna_pruner
    },
     {},
	 indicators[i]
   ]
     for i, exp in enumerate(experiments)
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(fine_tune_maco, params)
        for params in maco_fine_tuning_params_list
    )
    print(return_values)

### Running

In [ ]:
# Set the list of parameters
maco_run_params_list = [[
    {
        'name': f'MACO_{exp[0]}_{exp[1]}_{exp[2]}',
        'results_folder': results_folder,
        'fine_tuning_folder': tuning_folder,
        'num_runs': num_runs,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
     },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
     {
        'ker': maco_ker,
        'q': maco_q,
        'threshold': maco_threshold,
        'n_gen_mark': maco_n_gen_mark,
        'focus': maco_focus
     }
    ]
     for exp in experiments
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(run_maco, params)
        for params in maco_run_params_list
    )
    print(return_values)

## 4.4 MOPSO-CD

In [ ]:
#################### CUSTOMIZABLE ####################
# Tunable parameters
mopso_cd_w = 0.5
mopso_cd_c1 = 2.0
mopso_cd_c2 = 2.0
mopso_cd_max_velocity_rate = 0.5
mopso_cd_archive_size = population_size
######################################################

### Fine Tuning

In [ ]:
# Set the list of parameters
mopso_cd_fine_tuning_params_list = [[
    {
        'name': f'MOPSO_CD_{exp[0]}_{exp[1]}_{exp[2]}',
        'fine_tuning_folder': tuning_folder,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
    },
    get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
    {
        'n_trials': n_trials,
        'n_steps': n_steps,
        'pruner': optuna_pruner
    },
    {},
    indicators[i]
    ]
     for i, exp in enumerate(experiments)
]

# Execute in parallel
if __name__ == "__main__":
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(fine_tune_mopso_cd, params)
        for params in mopso_cd_fine_tuning_params_list
    )
    print(return_values)

### Running

In [ ]:
# Set the list of parameters
mopso_cd_run_params_list = [[
    {
        'name': f'MOPSO_CD_{exp[0]}_{exp[1]}_{exp[2]}',
        'results_folder': results_folder,
        'fine_tuning_folder': tuning_folder,
        'num_runs': num_runs,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
     },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
     {
        'w': mopso_cd_w,
        'c1': mopso_cd_c1,
        'c2': mopso_cd_c2,
        'max_velocity_rate': mopso_cd_max_velocity_rate,
        'archive_size': mopso_cd_archive_size
     }
    ]
     for exp in experiments
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(run_mopso_cd, params)
        for params in mopso_cd_run_params_list
    )
    print(return_values)

## 4.5 NSGA-II

In [ ]:
#################### CUSTOMIZABLE ####################
# Tunable parameters
nsga2_recombination_probability = 0.5
nsga2_eta_recombination = 20
nsga2_mutation_probability = 0.1
nsga2_eta_mutation = 20
######################################################

### Fine Tuning

In [ ]:
# Set the list of parameters
nsga2_fine_tuning_params_list = [[
    {
        'name': f'NSGA2_{exp[0]}_{exp[1]}_{exp[2]}',
        'fine_tuning_folder': tuning_folder,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
    },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
	 {
        'n_trials': n_trials,
        'n_steps': n_steps,
        'pruner': optuna_pruner
    },
     {},
	 indicators[i]
   ]
     for i, exp in enumerate(experiments)
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(fine_tune_nsga2, params)
        for params in nsga2_fine_tuning_params_list
    )
    print(return_values)

### Running

In [ ]:
# Set the list of parameters
nsga2_run_params_list = [[
    {
        'name': f'NSGA2_{exp[0]}_{exp[1]}_{exp[2]}',
        'results_folder': results_folder,
        'fine_tuning_folder': tuning_folder,
        'num_runs': num_runs,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
     },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
     {
        'recombination_probability': nsga2_recombination_probability,
        'eta_recombination': nsga2_eta_recombination,
        'mutation_probability': nsga2_mutation_probability,
        'eta_mutation': nsga2_eta_mutation
     }
    ]
     for exp in experiments
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(run_nsga2, params)
        for params in nsga2_run_params_list
    )
    print(return_values)

## 4.6 NSPSO

In [ ]:
#################### CUSTOMIZABLE ####################
# Tunable parameters
nspso_omega = 0.5
nspso_c1 = 2.0
nspso_c2 = 2.0
nspso_chi = 2.0
nspso_velocity_coefficient = 0.5
nspso_leader_selection_range = 5
nspso_diversity_mechanism = 'crowding distance'
######################################################

### Fine Tuning

In [ ]:
# Set the list of parameters
nspso_fine_tuning_params_list = [[
    {
        'name': f'NSPSO_{exp[0]}_{exp[1]}_{exp[2]}',
        'fine_tuning_folder': tuning_folder,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
    },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
	 {
        'n_trials': n_trials,
        'n_steps': n_steps,
        'pruner': optuna_pruner
    },
     {},
	 indicators[i]
   ]
     for i, exp in enumerate(experiments)
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(fine_tune_nspso, params)
        for params in nspso_fine_tuning_params_list
    )
    print(return_values)

### Running

In [ ]:
# Set the list of parameters
nspso_run_params_list = [[
    {
        'name': f'NSPSO_{exp[0]}_{exp[1]}_{exp[2]}',
        'results_folder': results_folder,
        'fine_tuning_folder': tuning_folder,
        'num_runs': num_runs,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
     },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
     {
        'w': nspso_omega,
        'c1': nspso_c1,
        'c2': nspso_c2,
        'chi': nspso_chi,
        'velocity_coefficient': nspso_velocity_coefficient,
        'leader_selection_range': nspso_leader_selection_range,
        'diversity_mechanism': nspso_diversity_mechanism
     }
    ]
     for exp in experiments
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(run_nspso, params)
        for params in nspso_run_params_list
    )
    print(return_values)

## 4.7 SPEA-2

In [ ]:
#################### CUSTOMIZABLE ####################
# Tunable parameters
spea2_recombination_probability = 0.5
spea2_eta_recombination = 20
spea2_mutation_probability = 0.1
spea2_eta_mutation = 20
######################################################

### Fine Tuning

In [ ]:
# Set the list of parameters
spea2_fine_tuning_params_list = [[
    {
        'name': f'SPEA2_{exp[0]}_{exp[1]}_{exp[2]}',
        'fine_tuning_folder': tuning_folder,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
    },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
	 {
        'n_trials': n_trials,
        'n_steps': n_steps,
        'pruner': optuna_pruner
    },
     {},
	 indicators[i]
   ]
     for i, exp in enumerate(experiments)
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(fine_tune_spea2, params)
        for params in spea2_fine_tuning_params_list
    )
    print(return_values)

### Running

In [ ]:
# Set the list of parameters
spea2_run_params_list = [[
    {
        'name': f'SPEA2_{exp[0]}_{exp[1]}_{exp[2]}',
        'results_folder': results_folder,
        'fine_tuning_folder': tuning_folder,
        'num_runs': num_runs,
        'max_fitness_eval': max_fitness_eval,
        'population_size': population_size,
        'random_state': random_state
     },
     get_exp_problem(exp[0], exp[1], exp[2], exp[3]),
     {
        'recombination_probability': spea2_recombination_probability,
        'eta_recombination': spea2_eta_recombination,
        'mutation_probability': spea2_mutation_probability,
        'eta_mutation': spea2_eta_mutation}
    ]
     for exp in experiments
]

# Execute in parallel
if __name__ == '__main__':
    return_values = Parallel(n_jobs=workers)(
        delayed(safe_run)(run_spea2, params)
        for params in spea2_run_params_list
    )
    print(return_values)